In [ ]:
from scipy.io import loadmat, savemat
import time
import os
import urllib.request as ur
import io
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from tqdm import tqdm

baseURL = "http://ieeg-swez.ethz.ch/long-term_dataset/"
basePath = "/mnt/DATA/NonLinearMI/iEEG"
appendix="info.mat"


# Download raw data

Download the first available hour without seizures (only missing files, about 11 GB in total)

In [ ]:
for i in range(1,19):
    out_fname=f"iEEG_ID{i:02}.mat"
    out_path = os.path.join(basePath, out_fname)
    if not os.path.isfile(out_path):
        sub = f"ID{i:02}"
        fname = "_".join([sub, appendix])
        URL = os.path.join(baseURL, sub, fname)
        request_data = ur.urlopen(URL)
        file_like = io.BytesIO(request_data.read())
        mat = loadmat(file_like)
        bad_hours = set()
        for p in zip(mat['seizure_begin'], mat['seizure_end']):
            for e in p:
                bad_hours.add(int(e[0]/3600))

        for j in range(100):
            if j not in bad_hours:
                data_fname = "_".join([sub, f"{j+1}h.mat"])
                mat_path = os.path.join(baseURL, sub, data_fname)
                print(i, mat_path)
                print(ur.urlretrieve(mat_path, out_path)[0])
                break

In [ ]:
out_fname=f"iEEG_info.csv"
out_path = os.path.join(basePath, out_fname)

if not os.path.isfile(out_path):
    frequencies=[]
    for i in range(1,19):
        sub = f"ID{i:02}"
        fname = "_".join([sub, appendix])
        URL = os.path.join(baseURL, sub, fname)
        request_data = ur.urlopen(URL)
        file_like = io.BytesIO(request_data.read())
        mat = loadmat(file_like)
        frequencies.append([i,mat['fs'][0,0]])
    subj_data = pd.DataFrame(frequencies, columns=["Subject", "Hz"]).set_index("Subject")
    subj_data.to_csv(out_path, index=True)
else:
    subj_data = pd.read_csv(out_path, index_col=0)

display(subj_data)


# Data overview

In [ ]:
out_fname=f"iEEG_info.csv"
out_path = os.path.join(basePath, out_fname)
if "Electrodes" not in subj_data:
    electrodes = []
    samples = []
    for i in range(1,19):
        out_fname=f"iEEG_ID{i:02}.mat"
        in_path = os.path.join(basePath, out_fname)
        mat = loadmat(in_path)
        print(i, mat['EEG'].shape)
        electrodes.append(mat['EEG'].shape[0])
        samples.append(mat['EEG'].shape[1])
    subj_data["Electrodes"]=electrodes
    subj_data["Samples"]=samples
    subj_data.to_csv(out_path, index=True)

subj_data.hist("Electrodes", bins="auto")
plt.title(subj_data.Electrodes.min())
plt.show()

# Downsampling

## The three methods are equivalent

In [ ]:
pID = 1
out_fname=f"iEEG_ID{pID:02}.mat"
in_path = os.path.join(basePath, out_fname)
mat = loadmat(in_path)
plt.plot(mat['EEG'][0,:512], label="Orig")

resampled, t = signal.resample(mat['EEG'][0,:512], 100, np.arange(512))
plt.plot(t, resampled, ".", label="Resampl")

decimated = signal.decimate(mat['EEG'][0,:512], 5)
plt.plot(5*np.arange(len(decimated)), decimated, "x", label="Decim")

# sampling frequency
f_sample = 512
# pass band frequency
f_pass = 44
# stop band frequency
f_stop = 70
# pass band ripple
g_pass = 0.5
# stop band attenuation
g_stop = 8.1
N, Wn = signal.buttord(f_pass, f_stop, g_pass, g_stop, analog=False, fs=f_sample)
print(N, Wn)
sos = signal.butter(N, Wn, 'lp', False, 'sos', f_sample) 
#double pass Butterworth
filtered = signal.sosfilt(sos, mat['EEG'][0,:512])
filtered = signal.sosfilt(sos, filtered[::-1])[::-1]
filter_resample, t = signal.resample(filtered, 100, np.arange(512))

plt.plot(filtered, ls="-.", lw=1, label="Butt")
plt.plot(np.arange(len(filtered),step=5),filtered[::5], "+k", label="Butt Skip")
plt.plot(t, filter_resample, "3y", label="Butt Res")
plt.legend()
plt.show()
# plt.plot(np.arange(512)-256,np.fft.fftshift(np.fft.fft(mat['EEG'][0,:512]))/4, label="Original")
# plt.plot(np.arange(512)-256,np.fft.fftshift(np.fft.fft(filtered))/4, label="Butterworth")
plt.plot(np.arange(100)-50,np.fft.fftshift(np.fft.fft(resampled)), label="Resampled")
decl = len(decimated)
plt.plot(np.arange(decl)-int(decl/2),np.fft.fftshift(np.fft.fft(decimated)), label="Decimated")
plt.plot(np.arange(decl)-int(decl/2),np.fft.fftshift(np.fft.fft(filtered[::5])), label="Butt skipping")
plt.plot(np.arange(100)-50,np.fft.fftshift(np.fft.fft(filter_resample)), label="Butt Resamp")
plt.ylim((-300, 300))
plt.legend()
plt.show()

In [ ]:
# sampling frequency
f_sample = 512
# pass band frequency
f_pass = 44
# stop band frequency
f_stop = 70
# pass band ripple
g_pass = 0.5
# stop band attenuation
g_stop = 8.1
N, Wn = signal.buttord(f_pass, f_stop, g_pass, g_stop, analog=False, fs=f_sample)
print(N, Wn)
sos = signal.butter(N, Wn, 'lp', False, 'sos', f_sample) 
fs_dest = 100
tot_samp_dest = 10000
sec_dest = int(tot_samp_dest / fs_dest)+1
tot_sec = 3600
tot_samp_orig = f_sample * (sec_dest+1) + 100
new_dataset = np.empty([tot_samp_dest, 24, 18])
for i in tqdm(range(1,19)):
    out_fname=f"iEEG_ID{i:02}.mat"
    in_path = os.path.join(basePath, out_fname)
    EEG = loadmat(in_path)['EEG']
    selected_electrodes = np.random.choice(EEG.shape[0], 24, False)
    filtered = signal.sosfilt(sos, signal.sosfilt(sos, EEG[selected_electrodes,:tot_samp_orig])[:,::-1])[:,-100:f_sample:-1]
    new_dataset[:,:,i-1]=signal.resample(filtered.T, sec_dest*fs_dest)[:tot_samp_dest,:]

savemat("/mnt/DATA/NonLinearMI/iEEG/iEEG_part.mat",{"EEG":new_dataset})